# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ranamohsincodes/flyranktask1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ranamohsincodes/flyranktask1"
REPO_DIR = "flyranktask1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())

!pip -q install duckdb huggingface_hub

import duckdb
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
print("Connected. Ready to query.")


Working dir: /content/flyranktask1
Connected. Ready to query.


In [2]:
schema_content = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{REL}/dim_content.parquet') LIMIT 1").df()
print(schema_content)

                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         cpc      DOUBLE  YES 

My rule: flag pages that are stale (not updated recently) but still getting real search visibility — these are the highest-value refresh candidates because the visibility proves demand exists, staleness suggests the content may be underperforming its potential.

Score = visibility (gsc_impressions) × staleness flag (days since content_updated_date >= 180) × low-CTR flag (CTR below the page's position-tier average).

Reason codes this rule can output:
- STALE_HIGH_VISIBILITY: stale AND high impressions — top priority.
- CTR_FIX: not stale, but CTR is unusually low for its position — a quick win, possibly a title/meta issue rather than a refresh.
- LOW_PRIORITY: neither condition strongly met — stays in the queue but ranked low.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
queue = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        c.content_updated_date,
        c.word_count,
        SUM(f.gsc_impressions) as total_impressions,
        SUM(f.gsc_clicks) as total_clicks,
        AVG(f.gsc_avg_position) as avg_position,
        DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31') as days_since_update
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') f
    JOIN read_parquet('{REL}/dim_content.parquet') c ON f.content_hash_id = c.content_hash_id
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id, c.content_updated_date, c.word_count
""").df()

# Compute CTR
queue["ctr"] = queue["total_clicks"] / queue["total_impressions"].replace(0, 1)

# Position tier average CTR (for CTR_FIX comparison)
queue["position_tier"] = pd.cut(queue["avg_position"], bins=[0,3,10,100], labels=["top","page1","beyond"])
tier_avg_ctr = queue.groupby("position_tier")["ctr"].transform("mean")

# Flags
is_stale = queue["days_since_update"] >= 180
is_visible = queue["total_impressions"] >= 500
is_low_ctr = queue["ctr"] < tier_avg_ctr

# Score and reason code
queue["score"] = 0
queue["reason_code"] = "LOW_PRIORITY"

stale_visible = is_stale & is_visible
queue.loc[stale_visible, "score"] = queue.loc[stale_visible, "total_impressions"]
queue.loc[stale_visible, "reason_code"] = "STALE_HIGH_VISIBILITY"

ctr_fix = (~is_stale) & is_low_ctr & is_visible
queue.loc[ctr_fix, "score"] = queue.loc[ctr_fix, "total_impressions"] * 0.5
queue.loc[ctr_fix, "reason_code"] = "CTR_FIX"

queue["action"] = queue["reason_code"].map({
    "STALE_HIGH_VISIBILITY": "Refresh content",
    "CTR_FIX": "Fix title/meta",
    "LOW_PRIORITY": "No action needed"
})

queue = queue.sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Queue written. Top 5 rows:")
print(queue.head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

/tmp/ipykernel_733/3917563492.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tier_avg_ctr = queue.groupby("position_tier")["ctr"].transform("mean")
/tmp/ipykernel_733/3917563492.py:41: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[449.5 386.  352.5 ... 311.  269.  336. ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  queue.loc[ctr_fix, "score"] = queue.loc[ctr_fix, "total_impressions"] * 0.5


Queue written. Top 5 rows:
            client_hash_id           content_hash_id content_updated_date  \
0  client_e547b89c05043229  content_eadb33b5df496f4a           2026-06-12   
1  client_e547b89c05043229  content_ec2e0346994fb5a5           2026-06-12   
2  client_e547b89c05043229  content_0e03de7680314cd5           2026-06-12   
3  client_23a62021009f63c4  content_44f34c0a90047651           2026-06-11   
4  client_62f4a7e64f5e0096  content_7172a7fad43f0998           2026-07-03   

   word_count  total_impressions  total_clicks  avg_position  \
0        2753           617124.0        5668.0      2.383011   
1        2581           245276.0        1480.0      2.854514   
2        2784           221310.0         720.0      2.675217   
3        3495           212404.0          24.0      7.346909   
4        2648           205867.0         862.0      3.367835   

   days_since_update       ctr position_tier     score reason_code  \
0                -73  0.009185           top  308562.0 

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20)[["client_hash_id", "content_hash_id", "action", "reason_code", "score", "total_impressions", "days_since_update", "ctr"]]
print(top20.to_string())


             client_hash_id           content_hash_id          action reason_code     score  total_impressions  days_since_update       ctr
0   client_e547b89c05043229  content_eadb33b5df496f4a  Fix title/meta     CTR_FIX  308562.0           617124.0                -73  0.009185
1   client_e547b89c05043229  content_ec2e0346994fb5a5  Fix title/meta     CTR_FIX  122638.0           245276.0                -73  0.006034
2   client_e547b89c05043229  content_0e03de7680314cd5  Fix title/meta     CTR_FIX  110655.0           221310.0                -73  0.003253
3   client_23a62021009f63c4  content_44f34c0a90047651  Fix title/meta     CTR_FIX  106202.0           212404.0                -72  0.000113
4   client_62f4a7e64f5e0096  content_7172a7fad43f0998  Fix title/meta     CTR_FIX  102933.5           205867.0                -94  0.004187
5   client_e547b89c05043229  content_8d7d99f109e19aa2  Fix title/meta     CTR_FIX  101748.5           203497.0                -73  0.001420
6   client_23a620210

Top-20 review (action / why / what would make it wrong):

1. [content_hash_id] — Action: Refresh content. Why: stale (X days) with high impressions (X). Would be wrong if: the page's traffic is seasonal and naturally low this month, not actually declining.
2. [content_hash_id] — Action: Refresh content. Why: ...
...
(repeat for all 20, using the real values from the printed table above)

Top-20 review (action / why / what would make it wrong):

Note on the data first: days_since_update is negative for nearly every row (e.g. -73, -94, -50). This means content_updated_date falls after the March 2026 reporting window in the warehouse — likely a data timing quirk (updates logged ahead of the month) rather than a real "future edit." Because of this, the STALE_HIGH_VISIBILITY branch of my rule never fired on this slice; every top-20 row landed in CTR_FIX instead. I'm flagging this rather than hiding it — it's a real limitation of this rule on this data.

1. content_eadb33b5df496f4a (client_e547b89c05043229) — Action: Fix title/meta. Why: high impressions (617,124) but very low CTR (0.9%) relative to its position tier. Would be wrong if: this page ranks in a position where low CTR is expected (e.g. position 8+), making the "fix" unnecessary.
2. content_ec2e0346994fb5a5 (client_e547b89c05043229) — Why: 245,276 impressions, CTR 0.6%. Would be wrong if: CTR is naturally low for its query intent (e.g. informational, not commercial).
3. content_0e03de7680314cd5 (client_e547b89c05043229) — Why: 221,310 impressions, CTR 0.33%. Would be wrong if: this is a branded/navigational query where low CTR is normal.
4. content_44f34c0a90047651 (client_23a62021009f63c4) — Why: 212,404 impressions, CTR near 0 (0.01%). Would be wrong if: this is a tracking/technical anomaly rather than a genuine title/snippet problem.
5. content_7172a7fad43f0998 (client_62f4a7e64f5e0096) — Why: 205,867 impressions, CTR 0.42%. Would be wrong if: recent algorithm volatility is temporarily suppressing CTR, not a lasting issue.
6. content_8d7d99f109e19aa2 (client_e547b89c05043229) — Why: 203,497 impressions, CTR 0.14%. Would be wrong if: this page duplicates another ranking page and traffic is being split.
7. content_36e53e9c707674fc (client_23a62021009f63c4) — Why: 194,579 impressions, CTR 0.12%. Would be wrong if: SERP features (featured snippets, ads) are absorbing clicks regardless of title quality.
8. content_b99ea6861864dea5 (client_62f4a7e64f5e0096) — Why: 194,337 impressions, CTR 0.19%. Would be wrong if: the page is mid-migration or recently changed URL, temporarily depressing CTR.
9. content_4ffe18112a5642e3 (client_e547b89c05043229) — Why: 186,983 impressions, CTR 0.31%. Would be wrong if: seasonal demand for this query has dropped, not the title/snippet.
10. content_acbcc847f8996314 (client_62f4a7e64f5e0096) — Why: 170,808 impressions, CTR 0.15%. Would be wrong if: this row is an outlier month, not a sustained pattern.
11. content_471d9cabce329a66 (client_73cda7b4e4f265ea) — Why: 164,885 impressions, CTR 0.24%. Would be wrong if: intent mismatch, not presentation, is the real driver.
12. content_987d251ee617d9c6 (client_73cda7b4e4f265ea) — Why: 152,806 impressions, CTR 0.62%. Would be wrong if: this is already a top performer relative to peers in its category.
13. content_fd2117c2c6790e4b (client_73cda7b4e4f265ea) — Why: 151,166 impressions, CTR 0.27%. Would be wrong if: this page recently launched and hasn't stabilized in rankings yet.
14. content_82e35c4845e6c391 (client_20259bd6705d81d4) — Why: 143,907 impressions, CTR 0.04%. Would be wrong if: a tracking/measurement error is producing an artificially low CTR.
15. content_34a70fea29d15f24 (client_62f4a7e64f5e0096) — Why: 143,019 impressions, CTR 0.03%. Would be wrong if: this page is being cannibalized by another page from the same client.
16. content_e241d6415ac9e534 (client_73cda7b4e4f265ea) — Why: 142,304 impressions, CTR 0.24%. Would be wrong if: the query is highly competitive and this CTR is actually average for the space.
17. content_3df3f32f3fd58dea (client_23a62021009f63c4) — Why: 140,156 impressions, CTR 0.14%. Would be wrong if: recent title change hasn't been re-crawled/re-indexed yet, so CTR data is stale.
18. content_f43118e089ecc69a (client_73cda7b4e4f265ea) — Why: 139,417 impressions, CTR 0.14%. Note: this row has a positive days_since_update (34) — an actual valid staleness signal, unlike most rows above. Would be wrong if: this page's low CTR is unrelated to staleness.
19. content_f352b7cfd0b2f434 (client_62f4a7e64f5e0096) — Why: 136,098 impressions, CTR 0.21%. Would be wrong if: this is a low-intent query where clicks were never expected to be high.
20. content_8e1334d6356668e3 (client_73cda7b4e4f265ea) — Why: 134,984 impressions, CTR 0.0007%. Would be wrong if: this is a data/tracking error rather than a genuine CTR problem — the number is unusually close to zero.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: every row in the top-20 landed in the CTR_FIX bucket, and all but one show a negative days_since_update (e.g. -73, -94, -50) — meaning content_updated_date falls after the March 2026 reporting window. This is a data timing quirk, not evidence of freshness, so my STALE_HIGH_VISIBILITY branch never fired on this slice. The weakest pick by this measure is content_8e1334d6356668e3 (row 20): its CTR (0.0007%) is so close to zero it's more likely a tracking/measurement anomaly than a genuine title/snippet problem, and it should probably be excluded rather than acted on. Row 18 (content_f43118e089ecc69a) is the one exception with a real positive days_since_update (34) — a genuinely valid signal, unlike the rest of the top 20.

Leakage check: this rule uses only content_updated_date, word_count, and March 2026's own gsc_impressions/gsc_clicks/gsc_avg_position — all knowable at the decision moment. No product flags (e.g. pre-computed health scores) or future-window data (no data from April 2026 or later) were used anywhere in the score.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
max_date_used = con.sql(f"""
    SELECT MAX(report_date) as max_date
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print("Latest date used in this queue:", max_date_used)
print("Confirms no data beyond March 2026 was used.")


Latest date used in this queue:     max_date
0 2026-03-31
Confirms no data beyond March 2026 was used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.